# The classic NLP pipeline — a sentiment classifier



_Capstones_



**Walk the path that took natural-language processing from bag-of-words to attention: build a sentiment classifier out of word embeddings, a bidirectional LSTM, and an attention-pooling head.**



---



This notebook builds the pipeline one stage at a time — toy data, tokenizer, model, training loop, and finally the attention weights that show *where* the model read the sentiment. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.

import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — Generate a toy sentiment dataset

We synthesise our own labelled sentences so the task is controlled. Each sentence is mostly **neutral filler** with one or two **sentiment words** dropped in at random positions. Because sentence length and filler are the same for both classes, the model can't cheat on length — it *must* learn to find the sentiment word.

In [ ]:
import torch
import torch.nn as nn
import random

torch.manual_seed(1)
random.seed(1)

# Word pools: positive, negative, and neutral filler.
pos_words = ["great","love","excellent","amazing","wonderful","fantastic",
             "brilliant","superb","delightful","gripping"]
neg_words = ["terrible","awful","horrible","boring","dreadful","disappointing",
             "tedious","lousy","painful","forgettable"]
neutral   = ["the","a","this","that","movie","film","story","plot","scene","cast",
             "i","we","it","was","is","really","saw","watched","with","and","of",
             "to","on","very","quite","about","some","one","first","last"]

def make(label, n):
    out = []
    for _ in range(n):
        L = random.randint(7, 12)                          # random sentence length
        words = [random.choice(neutral) for _ in range(L)] # start with all filler
        pool = pos_words if label == 1 else neg_words
        for _ in range(random.randint(1, 2)):              # inject 1-2 sentiment words
            words[random.randrange(L)] = random.choice(pool)
        out.append((" ".join(words), label))
    return out

train = make(1, 120) + make(0, 120)
test  = make(1, 30)  + make(0, 30)
random.shuffle(train)

### Step 2 — Tokenize and pack into padded tensors

We split each sentence on spaces and build a `word -> id` vocabulary (id `0` is reserved for `<pad>`). Sentences have different lengths, so `batchify` pads every sentence in a batch to the longest one and returns a **mask** marking the real tokens — the attention layer later uses that mask to ignore padding.

In [ ]:
# TOKENIZE: split on spaces, build a word -> id vocabulary.
def tok(s):
    return s.split()

vocab = {"<pad>": 0}
for s, _ in train + test:
    for w in tok(s):
        if w not in vocab:
            vocab[w] = len(vocab)
V = len(vocab)

def encode(s):
    return [vocab[w] for w in tok(s)]

def batchify(data):                       # pad to the longest sentence, build a mask
    seqs = [encode(s) for s, _ in data]
    L = max(len(x) for x in seqs)
    X = torch.zeros(len(seqs), L, dtype=torch.long)
    mask = torch.zeros(len(seqs), L)
    for i, s in enumerate(seqs):
        X[i, :len(s)] = torch.tensor(s)   # real tokens
        mask[i, :len(s)] = 1              # 1 where there's a real token, 0 for padding
    y = torch.tensor([lab for _, lab in data])
    return X, mask, y

Xtr, Mtr, ytr = batchify(train)
Xte, Mte, yte = batchify(test)

### Step 3 — Define the model: embed → BiLSTM → attention → classify

The pipeline has four parts. An **embedding** turns each word id into a vector. A **bidirectional LSTM** reads the sentence both ways, producing a context vector per token. The **attention** layer scores each token, softmaxes the scores into weights (after masking padding to `-1e9`), and collapses the sentence into **one** weighted vector. A linear **classifier** turns that vector into positive/negative logits. We return the attention weights too, so we can inspect them.

In [ ]:
EMB, HID = 32, 48

class SentimentNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb  = nn.Embedding(V, EMB, padding_idx=0)        # word -> vector table
        self.lstm = nn.LSTM(EMB, HID, batch_first=True,
                            bidirectional=True)                # BiLSTM encoder
        self.attn = nn.Linear(2 * HID, 1)                      # one score per token
        self.cls  = nn.Linear(2 * HID, 2)                      # classifier head

    def forward(self, x, mask):
        embedded = self.emb(x)                                 # (B, L, EMB)
        h, _ = self.lstm(embedded)                             # (B, L, 2H) context per token
        score = self.attn(h).squeeze(-1)                       # (B, L) one score per token
        score = score.masked_fill(mask == 0, -1e9)             # ignore padding positions
        alpha = torch.softmax(score, dim=1)                    # weights over tokens, sum to 1
        ctx = (alpha.unsqueeze(-1) * h).sum(1)                 # ONE sentence vector
        logits = self.cls(ctx)
        return logits, alpha                                   # logits + attention weights

net   = SentimentNet()
opt   = torch.optim.Adam(net.parameters(), lr=4e-3, weight_decay=1e-4)
lossf = nn.CrossEntropyLoss()

### Step 4 — Train the classifier

Standard mini-batch training: for 80 epochs we shuffle the training set, take batches of 32, compute the cross-entropy loss on the logits, and step the Adam optimizer. We discard the attention weights here (`_`) since training only needs the logits.

In [ ]:
for ep in range(80):
    perm = torch.randperm(len(ytr))                # fresh shuffle each epoch
    for i in range(0, len(ytr), 32):               # mini-batches of 32
        idx = perm[i:i + 32]
        logits, _ = net(Xtr[idx], Mtr[idx])
        loss = lossf(logits, ytr[idx])
        opt.zero_grad()
        loss.backward()
        opt.step()

### Step 5 — Evaluate and inspect the attention

First we measure **test accuracy** in eval mode with gradients off. Then we feed one hand-written sentence through the model and print the **attention weight** on each word as a bar chart. The largest weights should land on the sentiment-bearing words (`amazing`, `gripping`) — evidence the model learned to read sentiment *where it lives*, not in the filler.

In [ ]:
# Result 1: test accuracy.
net.eval()
with torch.no_grad():
    preds = net(Xte, Mte)[0].argmax(1)
    acc = (preds == yte).float().mean().item()
print("test accuracy:", round(acc, 3))            # our run: 0.917

# Result 2: attention weights highlight the sentiment words.
sent = "i watched the film and it was really amazing and gripping"
x = torch.tensor([encode(sent)])
m = torch.ones_like(x, dtype=torch.float)
with torch.no_grad():
    logits, alpha = net(x, m)

print("sentence:", sent)
print("prediction:", "positive" if logits.argmax(1).item() == 1 else "negative")
for w, a in zip(tok(sent), alpha[0].tolist()):
    bar = "#" * int(round(a * 50))
    print(f"  {w:10s} {a:.3f} {bar}")
# Our run: the two largest weights land on 'gripping' (0.237) and 'amazing'
# (0.136) -- the model learned to read sentiment WHERE it lives, not the filler.

## Visualize the data & results

_Does the finished pipeline classify sentiment correctly, and do its attention weights actually land on the sentiment-bearing words?_

### Step 1 — Run one labelled sentence through the trained model

Reusing the already-trained `net`, we encode a single sentence and do a forward pass with gradients off. The model returns both the classification `logits` and the per-token attention weights `alpha`.

In [ ]:
# After training, read off the attention weights over one labeled sentence. (PyTorch, seed 1.)
sent = "i watched the film and it was really amazing and gripping"
x = torch.tensor([encode(sent)])
m = torch.ones_like(x, dtype=torch.float)
with torch.no_grad():
    logits, alpha = net(x, m)

### Step 2 — Print the prediction and per-token weights

We print the predicted label and the attention weight on every token. The highest weights fall on `amazing` and `gripping` — the model concentrates its attention on exactly the words that carry the sentiment.

In [ ]:
print("prediction:", "positive" if logits.argmax(1).item() == 1 else "negative")
print("tokens :", tok(sent))
print("alpha  :", [round(a, 3) for a in alpha[0].tolist()])
# prediction: positive
# tokens : ['i','watched','the','film','and','it','was','really','amazing','and','gripping']
# alpha  : [0.029, 0.035, 0.037, 0.056, 0.065, 0.076, 0.092, 0.091, 0.136, 0.146, 0.237]
# Highest weights -> 'gripping' (0.237) and 'amazing' (0.136). test accuracy: 0.917